In [ ]:
import matplotlib.pyplot as plt

import numpy as np
import ROOT

import emm
import emm.coverage as cov

In [ ]:
# Configuration
n_toys = 400
n_bootstraps = 1000
n_events_per_toy = 5036

# Test Config
# n_toys = 2
# n_bootstraps = 2
# n_events_per_toy = 5036

np.random.seed(42)
seeds = np.random.randint(0, 10000, size=n_toys)

n_test_points = 1000
test_range = (500, 3500)
test_points = np.linspace(test_range[0], test_range[1], n_test_points)

In [ ]:
# Load the data
data_tree = emm.get_data(sort_and_index=True, tree=True)
x = ROOT.RooRealVar("x", "Diphoton Mass [GeV]", 500, 10_000)
data = ROOT.RooDataSet("mgg", "mgg", ROOT.RooArgSet(x), ROOT.RooFit.Import(data_tree))
n = data.numEntries()

In [ ]:
# Set up models
toy_models = [
    emm.f1(x, prefix="true"),
    emm.f2(x, prefix="true"),
    emm.f3(x, prefix="true"),
    emm.f4(x, prefix="true"),
]

model_primitives = [
    emm.f1,
    emm.f2,
    emm.f3,
    emm.f4,
    emm.make_model_primitive(emm.ExponentialMixtureModel, 2, data_mean=700, name="ExponentialMixture-2"),
    emm.make_model_primitive(emm.ExponentialMixtureModel, 3, data_mean=700, name="ExponentialMixture-3"),
    # emm.make_model_primitive(emm.ExponentialMixtureModel, 4, data_mean=700, name="ExponentialMixture-4"),
]


for toy_model in toy_models:
    print(f"Fitting toy model: {toy_model.name}")
    toy_model.pdf.fitTo(data)

true_pdf_vals = {
    toy_model.name: emm.evaluate_pdf(x, toy_model, test_points)
    for toy_model in toy_models
}

In [ ]:
# Run jobs
# cov.run_coverage_tasks(
#     x, toy_models, model_primitives,
#     seeds, n_bootstraps,
#     n_events_per_toy, test_points,
#     use_condor=True,
#     # remake=True,
# )

In [ ]:
coverages = cov.get_coverage_results(
    toy_models,
    seeds,
    n_bootstraps,
    n_events_per_toy,
    true_pdf_vals,
    alpha=0.05,
)

In [ ]:
# Plot the results
cov.plot_coverages(
    coverages, test_points,
    alpha=0.05, y_min=0.8,
    range=(500,3500),
    skip_every=20
)